# Notebook 10 — Discrete Mathematics, Graphs, and Combinatorics

**Module:** 02 — Mathematics for Biology  
**Notebook:** 10 of 12  
**Prerequisites:** NB01 (Python), Module 01 complete  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

Not all mathematical biology is about differential equations. Genomes are discrete:
4 nucleotides, sequences of finite length, exons that can be counted, SNPs that can
be enumerated. Protein interaction networks, metabolic pathways, and gene regulatory
networks are *graphs*. K-mer counting — the foundation of de Bruijn assembly and many
alignment-free methods — is combinatorics. This notebook covers the discrete
mathematics that shows up throughout Modules 08–14.

---
## Step 2 — Intuition

**Combinatorics** counts: how many ways can something be arranged?
**Graphs** represent relationships: nodes are objects, edges are connections.

A genome is a sequence over an alphabet of size 4. A k-mer is a window of length $k$.
Counting k-mers, finding unique sequences, comparing distributions across genomes —
all of this is combinatorics and string algorithms running on a graph (the de Bruijn
graph) under the hood.

---
## Step 3 — Biological Background

**K-mers in genomics:**
- A k-mer is a substring of length $k$ extracted from a DNA sequence
- Every genome of length $L$ contains $L - k + 1$ overlapping k-mers
- Total possible k-mers: $4^k$ (alphabet of size 4)
- Genome assemblers build a **de Bruijn graph**: nodes are (k-1)-mers, edges are k-mers

**Protein interaction networks:**
- Nodes = proteins; edges = physical interactions
- Hub proteins (high degree) are often essential genes
- Network topology: scale-free distributions observed in many PPI networks

**Sequence counting and the birthday problem:**
How long must a random sequence be before we expect to see every possible k-mer at
least once? This is the *coupon collector's problem* — a combinatorial result with
direct implications for sequencing depth requirements.

---
## Step 4 — Mathematical Explanation

**Combinatorics fundamentals:**

| Concept | Formula | DNA example |
|---------|---------|-------------|
| Total k-mers over alphabet Σ | $|\Sigma|^k$ | $4^k$ possible k-mers |
| Permutations of n items | $n!$ | Orderings of n genes |
| Combinations (choose k from n) | $\binom{n}{k} = \frac{n!}{k!(n-k)!}$ | Number of ways to choose k SNPs from n |
| Expected unique k-mers in genome of length L | $L - k + 1$ (with repeats) | Raw count |

**Graph theory basics:**
- $G = (V, E)$ where $V$ = vertices, $E$ = edges
- **Degree** $d(v)$: number of edges incident to vertex $v$
- **Path**: sequence of vertices connected by edges
- **Connected component**: maximal set of vertices with paths between all pairs
- **Adjacency matrix** $A$: $A_{ij} = 1$ if edge $(i,j)$ exists

**Degree distribution:** $P(k)$ = fraction of nodes with degree $k$.
- Random graph (Erdős-Rényi): Poisson distribution
- Scale-free network: power law $P(k) \sim k^{-\gamma}$

---
## Step 5 — Computational Explanation

**Python tools:**
- `collections.Counter` — k-mer counting in O(L)
- `itertools.product` — enumerate all possible k-mers
- `networkx` — graph construction, degree computation, path finding, visualization
- `numpy` — adjacency matrix operations

**K-mer counting algorithm:**
```
for i in range(len(seq) - k + 1):
    kmer = seq[i:i+k]
    counts[kmer] += 1
```
Time: O(L), Space: O(4^k) worst case.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import networkx as nx
from collections import Counter
from itertools import product
import matplotlib.pyplot as plt
from math import comb, factorial

In [ ]:
# Cell 6.1 — Combinatorics: counting DNA sequences
k = 4
alphabet_size = 4
total_kmers = alphabet_size ** k
print(f"Total possible {k}-mers: 4^{k} = {total_kmers}")

# Enumerate all 4-mers
all_kmers = [''.join(c) for c in product('ACGT', repeat=k)]
print(f"First 8: {all_kmers[:8]}")
print(f"Total enumerated: {len(all_kmers)}")

# Combinations: choose 3 SNPs from 10
n_snps, k_choose = 10, 3
print(f"\nWays to choose {k_choose} SNPs from {n_snps}: C({n_snps},{k_choose}) = {comb(n_snps, k_choose)}")

In [ ]:
# Cell 6.2 — K-mer counting from a sequence
def count_kmers(seq: str, k: int) -> Counter:
    """
    Count all k-mers in seq (sliding window, O(L) time).

    Parameters
    ----------
    seq : str — DNA sequence (uppercase ACGT)
    k   : int — k-mer length

    Returns
    -------
    Counter mapping k-mer string to count
    """
    return Counter(seq[i:i+k] for i in range(len(seq) - k + 1))

# Test on a simple sequence
seq = "ATGATGATG"
counts_3 = count_kmers(seq, 3)
print(f"Sequence: {seq}")
print(f"3-mer counts: {dict(counts_3)}")

# On a random genome of length 1000
rng = np.random.default_rng(42)
genome = ''.join(rng.choice(list('ACGT'), size=1000))
counts_4 = count_kmers(genome, 4)
print(f"\nRandom genome (L=1000), 4-mer statistics:")
print(f"  Unique 4-mers observed: {len(counts_4)} / {4**4} possible")
print(f"  Most common: {counts_4.most_common(3)}")
print(f"  Expected count per k-mer: {(1000 - 4 + 1) / 4**4:.1f}")

In [ ]:
# Cell 6.3 — Protein interaction network with networkx
# Small synthetic PPI network
ppi = nx.barabasi_albert_graph(n=50, m=2, seed=42)  # scale-free graph

# Relabel as protein names
protein_names = {i: f"P{i+1:02d}" for i in range(50)}
ppi = nx.relabel_nodes(ppi, protein_names)

# Basic graph statistics
degrees = dict(ppi.degree())
degree_values = list(degrees.values())

print(f"Network statistics:")
print(f"  Nodes (proteins): {ppi.number_of_nodes()}")
print(f"  Edges (interactions): {ppi.number_of_edges()}")
print(f"  Average degree: {np.mean(degree_values):.2f}")
print(f"  Max degree (hub): {max(degree_values)} ({max(degrees, key=degrees.get)})")
print(f"  Connected: {nx.is_connected(ppi)}")
# Shortest path example
p1, p2 = "P01", "P25"
if nx.has_path(ppi, p1, p2):
    path = nx.shortest_path(ppi, p1, p2)
    print(f"  Shortest path {p1}→{p2}: {path} (length {len(path)-1})")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — K-mer frequency distribution and network
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: K-mer count distribution
ax = axes[0]
count_vals = list(counts_4.values())
ax.hist(count_vals, bins=15, color="steelblue", edgecolor="white")
ax.axvline(np.mean(count_vals), color="tomato", ls="--", label=f"Mean = {np.mean(count_vals):.1f}")
ax.set_xlabel("K-mer count"); ax.set_ylabel("Number of k-mers")
ax.set_title("4-mer frequency distribution (L=1000)")
ax.legend()

# Panel 2: Degree distribution of PPI network
ax = axes[1]
degree_count = Counter(degree_values)
k_vals = sorted(degree_count.keys())
pk_vals = [degree_count[k_]/len(degree_values) for k_ in k_vals]
ax.scatter(k_vals, pk_vals, color="steelblue", zorder=5)
ax.set_xlabel("Degree k"); ax.set_ylabel("P(k)")
ax.set_title("PPI network degree distribution")
ax.set_yscale("log"); ax.set_xscale("log")

# Panel 3: Network visualisation (small subgraph)
ax = axes[2]
hub_node = max(degrees, key=degrees.get)
subgraph = ppi.subgraph([hub_node] + list(ppi.neighbors(hub_node)))
pos = nx.spring_layout(subgraph, seed=42)
node_sizes = [200 + 80 * degrees[n] for n in subgraph.nodes()]
nx.draw_networkx(subgraph, pos=pos, ax=ax, node_size=node_sizes,
                 node_color=["tomato" if n == hub_node else "steelblue" for n in subgraph.nodes()],
                 font_size=6, edge_color="gray", alpha=0.9)
ax.set_title(f"Hub neighbourhood ({hub_node})")
ax.axis("off")

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. How many unique 6-mers are possible over the DNA alphabet? How many would you
   expect to observe in a genome of length 3 × 10⁶ bp? Assume uniform random sequence.
2. Implement `count_kmers` and run it on the BRCA1 sequence from Module 01's datasets.
   What are the top 5 most common 4-mers? Do any differ significantly from uniform expectation?
3. Build a de Bruijn graph: given a set of k-mers, construct a directed graph where
   each node is a (k-1)-mer and each edge corresponds to a k-mer (its overlap).
   Use `networkx.DiGraph`. Test on the sequence `ATGCATGC` with k=3.
4. For the Barabási-Albert graph in Cell 6.3: compute the clustering coefficient
   (`nx.average_clustering`). Is it higher or lower than an Erdős-Rényi graph with
   the same number of nodes and edges? What does this mean biologically?

---
## Quiz — Active Recall

1. How many possible 10-mers are there over the DNA alphabet? Compute mentally.
2. What is a k-mer? What is a de Bruijn graph node and edge in terms of k-mers?
3. What is the degree of a node in a protein interaction network? What is a hub?
4. Write the formula for 'n choose k'. Apply it: how many ways to choose 2 mutations from 5 sites?
5. What is the difference between a path and a cycle in a graph?

---
## Reflection

**Date completed:** ____________________

> *[Can you explain a de Bruijn graph to a biologist who has never heard of it? What is k-mer counting used for in practice?]*

---
*Next: `11_optimization_gradient_descent.ipynb`*